# Phase B0 — one-time Colab recovery execution

This notebook accepts only the separately authorized execution bundle. It writes persistent artifacts to Google Drive. If interrupted, preserve the `.incomplete` directory and do not rerun without an audit.


## 1. Confirm the exact Tesla T4 runtime


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
if gpu_name != 'Tesla T4':
    raise RuntimeError(f'Frozen recovery requires Tesla T4, found {gpu_name!r}.')


## 2. Upload exactly one authorized recovery-execution bundle


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.tar.gz')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one .tar.gz recovery-execution bundle.')
ARCHIVE = archives[0]
print('Uploaded:', ARCHIVE)


## 3. Restore and verify the exact authorized source


In [ ]:
import json, os, pathlib, shutil, subprocess, sys, tarfile
extract_root = pathlib.Path('/content/phase_b0_recovery_execution_bundle')
repo_root = pathlib.Path('/content/latent-stroke-dynamics')
if extract_root.exists() or repo_root.exists():
    raise RuntimeError('Execution extraction already exists; use a fresh runtime.')
extract_root.mkdir()
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(extract_root)
manifest = json.loads((extract_root / 'bundle_manifest.json').read_text())
if manifest['status'] != 'phase_b0_colab_recovery_execution_bundle_authorized_once':
    raise RuntimeError('Unexpected execution bundle status.')
if manifest['recovery_authorized'] is not True or manifest['maximum_completed_executions'] != 1:
    raise RuntimeError('Execution bundle lacks exact one-time authorization.')
if manifest['formal_authorized'] is not False or manifest['phase_b1_authorized'] is not False or manifest['phase_b2_authorized'] is not False:
    raise RuntimeError('A later experimental phase was unexpectedly authorized.')
subprocess.run(['git', 'clone', '--branch', manifest['branch'], str(extract_root / 'repository.bundle'), str(repo_root)], check=True)
head = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
if head != manifest['source_commit']:
    raise RuntimeError('Restored Git commit does not match execution bundle manifest.')
for source in (extract_root / 'resources').rglob('*'):
    if source.is_file():
        destination = repo_root / source.relative_to(extract_root / 'resources')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
print(json.dumps(manifest, indent=2))


## 4. Install and run the exact authorized suite


In [ ]:
%cd /content/latent-stroke-dynamics
%pip install -q -e ".[dev]"
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


Expected: **145 passed**. Any failure stops before Drive mounting or recovery output.


## 5. Mount Google Drive and run the fail-closed readiness check


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
artifact_root = pathlib.Path('/content/drive/MyDrive/latent-stroke-dynamics-phase-b0-recovery')
if str(artifact_root) != manifest['artifact_root']:
    raise RuntimeError('Mounted artifact root differs from the authorization.')
readiness_path = pathlib.Path('/content/phase-b0-colab-recovery-readiness.json')
subprocess.run([sys.executable, 'experiments/26_phase_b_colab_recovery_execution_check.py', '--artifact-root', str(artifact_root), '--report', str(readiness_path)], check=True)
readiness = json.loads(readiness_path.read_text())
if readiness['status'] != 'phase_b0_colab_recovery_execution_ready_authorized_once':
    raise RuntimeError('Recovery readiness check did not pass.')
print(json.dumps(readiness, indent=2, sort_keys=True))


## 6. Explicit one-time execution switch
Do not change this switch until the exact bundle, 145 tests, Drive mount, and readiness report have all passed. Once execution starts, never start a second copy.


In [ ]:
RUN_AUTHORIZED_RECOVERY = False
if RUN_AUTHORIZED_RECOVERY is not True:
    raise RuntimeError('Execution remains paused. Change RUN_AUTHORIZED_RECOVERY to True only for the authorized run.')


## 7. Execute once, with a persistent Drive console log


In [ ]:
artifact_root.mkdir(parents=True, exist_ok=True)
log_path = artifact_root / 'phase-b0-colab-recovery-console.log'
if log_path.exists():
    raise RuntimeError(f'Console log already exists: {log_path}. Do not start a second run.')
command = [sys.executable, 'experiments/23_phase_b_colab_recovery.py', '--recovery', '--artifact-root', str(artifact_root)]
environment = dict(os.environ)
environment['PYTHONUNBUFFERED'] = '1'
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Recovery exited with code {return_code}. Preserve Drive artifacts and do not rerun.')


## 8. Verify final artifacts and download the small completion handoff


In [ ]:
final_root = artifact_root / 'phase-b0-joint-embedding-development-2026-08-24-colab-recovery'
incomplete_root = artifact_root / 'phase-b0-joint-embedding-development-2026-08-24-colab-recovery.incomplete'
if not final_root.is_dir() or incomplete_root.exists():
    raise RuntimeError('Recovery did not atomically finalize; preserve everything and do not rerun.')
decision = json.loads((final_root / 'decision.json').read_text())
run_config = json.loads((final_root / 'run_config.json').read_text())
integrity = json.loads((final_root / 'integrity_manifest.json').read_text())
journal = json.loads((final_root / 'recovery_stage_journal.json').read_text())
handoff = {
    'status': 'phase_b0_colab_recovery_execution_complete',
    'source_commit': head,
    'bundle_manifest': manifest,
    'final_root': str(final_root),
    'decision': decision,
    'run_config': run_config,
    'integrity_manifest': integrity,
    'journal': journal,
    'console_log': str(log_path),
    'do_not_rerun': True,
}
handoff_path = pathlib.Path('/content/phase-b0-colab-recovery-completion-handoff.json')
handoff_path.write_text(json.dumps(handoff, indent=2) + '\n')
print(json.dumps(handoff, indent=2, sort_keys=True))
files.download(str(handoff_path))
